# Northwind Sales Analysis (SQL)
I wanted to see where a food distributor's money actually comes from, who the key customers are, and where the risks sit. Eight questions, all answered in SQL on the Northwind sample database.

## 0. Setup
This grabs the Northwind SQLite file and cleans it up. The copy I used has about 15,000 fake orders and 2 junk customers added on top of the real data, so I delete those first. What's left is the original 830 orders from July 2016 to May 2018.

In [1]:
import sqlite3
import pandas as pd

# Download the Northwind SQLite database (free sample data, about 25 MB)
!curl -L -o northwind.db https://github.com/jpwhite3/northwind-SQLite3/raw/main/dist/northwind.db

conn = sqlite3.connect("northwind.db")
cur = conn.cursor()

# Keep only the original Northwind orders (OrderID 10248 to 11077)
cur.execute('DELETE FROM "Order Details" WHERE OrderID > 11077')
cur.execute("DELETE FROM Orders WHERE OrderID > 11077")
conn.commit()

# Check: expect 830 orders and 2155 order lines
print(cur.execute("SELECT COUNT(*) FROM Orders").fetchone())
print(cur.execute('SELECT COUNT(*) FROM "Order Details"').fetchone())

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 23.5M  100 23.5M    0     0  7690k      0  0:00:03  0:00:03 --:--:-- 14.6M
(830,)
(2155,)


## Q1. Which product categories bring in the most revenue?
Revenue = unit price x quantity x (1 - discount). Each order line links to a product, and each product links to a category.

In [2]:
query = """
SELECT c.CategoryName,
       ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 0) AS revenue
FROM "Order Details" od
JOIN Products p   ON p.ProductID = od.ProductID
JOIN Categories c ON c.CategoryID = p.CategoryID
GROUP BY c.CategoryName
ORDER BY revenue DESC
"""
pd.read_sql(query, conn)

,CategoryName,revenue
0,Beverages,267868.0
1,Dairy Products,234507.0
2,Confections,167357.0
3,Meat/Poultry,163022.0
4,Seafood,131262.0
5,Condiments,106047.0
6,Produce,99985.0
7,Grains/Cereals,95745.0


## Q2a. Who are the top 10 customers?
Same idea, but the chain goes order line -> order -> customer.

In [3]:
query = """
SELECT cu.CompanyName,
       ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 0) AS revenue
FROM "Order Details" od
JOIN Orders o     ON o.OrderID = od.OrderID
JOIN Customers cu ON cu.CustomerID = o.CustomerID
GROUP BY cu.CompanyName
ORDER BY revenue DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,CompanyName,revenue
0,QUICK-Stop,110277.0
1,Ernst Handel,104875.0
2,Save-a-lot Markets,104362.0
3,Rattlesnake Canyon Grocery,51098.0
4,Hungry Owl All-Night Grocers,49980.0
5,Hanari Carnes,32841.0
6,Königlich Essen,30908.0
7,Folk och fä HB,29568.0
8,Mère Paillarde,28872.0
9,White Clover Markets,27364.0


## Q2b. Which countries bring in the most?

In [4]:
query = """
SELECT cu.Country,
       ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 0) AS revenue
FROM "Order Details" od
JOIN Orders o     ON o.OrderID = od.OrderID
JOIN Customers cu ON cu.CustomerID = o.CustomerID
GROUP BY cu.Country
ORDER BY revenue DESC
LIMIT 10
"""
pd.read_sql(query, conn)

,Country,revenue
0,USA,245585.0
1,Germany,230285.0
2,Austria,128004.0
3,Brazil,106926.0
4,France,81358.0
5,UK,58971.0
6,Venezuela,56811.0
7,Sweden,54495.0
8,Canada,50196.0
9,Ireland,49980.0


## Q3. How is monthly revenue trending?
Uses a CTE plus `LAG` to compare each month to the one before. I left out May 2018 because the data stops on 6 May, so it would look like a crash when it's really just a partial month.

In [5]:
query = """
WITH monthly AS (
    SELECT strftime('%Y-%m', o.OrderDate) AS month,
           ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 0) AS revenue
    FROM "Order Details" od
    JOIN Orders o ON o.OrderID = od.OrderID
    WHERE o.OrderDate < '2018-05-01'          -- exclude the partial month
    GROUP BY month
)
SELECT month,
       revenue,
       LAG(revenue) OVER (ORDER BY month) AS prev_month,
       ROUND((revenue - LAG(revenue) OVER (ORDER BY month)) * 100.0
             / LAG(revenue) OVER (ORDER BY month), 1) AS growth_pct
FROM monthly
ORDER BY month
"""
pd.read_sql(query, conn)

,month,revenue,prev_month,growth_pct
0,2016-07,27862.0,NaN,NaN
1,2016-08,25485.0,27862.0,-8.5
2,2016-09,26381.0,25485.0,3.5
3,2016-10,37516.0,26381.0,42.2
4,2016-11,45600.0,37516.0,21.5
5,2016-12,45240.0,45600.0,-0.8
6,2017-01,61258.0,45240.0,35.4
7,2017-02,38484.0,61258.0,-37.2
8,2017-03,38547.0,38484.0,0.2
9,2017-04,53033.0,38547.0,37.6


## Q4. What's the best-selling product in each category?
Two CTEs back to back, then `RANK()` within each category.

In [6]:
query = """
WITH product_rev AS (
    SELECT c.CategoryName,
           p.ProductName,
           ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 0) AS revenue
    FROM "Order Details" od
    JOIN Products p   ON p.ProductID = od.ProductID
    JOIN Categories c ON c.CategoryID = p.CategoryID
    GROUP BY c.CategoryName, p.ProductName
),
ranked AS (
    SELECT *,
           RANK() OVER (PARTITION BY CategoryName ORDER BY revenue DESC) AS rnk
    FROM product_rev
)
SELECT CategoryName, ProductName, revenue
FROM ranked
WHERE rnk = 1
ORDER BY revenue DESC
"""
pd.read_sql(query, conn)

,CategoryName,ProductName,revenue
0,Beverages,Côte de Blaye,141397.0
1,Meat/Poultry,Thüringer Rostbratwurst,80369.0
2,Dairy Products,Raclette Courdavault,71156.0
3,Confections,Tarte au sucre,47235.0
4,Grains/Cereals,Gnocchi di nonna Alice,42593.0
5,Produce,Manjimup Dried Apples,41820.0
6,Seafood,Carnarvon Tigers,29172.0
7,Condiments,Vegie-spread,16701.0


## Q5. How do the employees compare?
`COUNT(DISTINCT ...)` makes sure each order is only counted once. I added revenue per order so you can tell who wins on volume and who wins on order size.

In [7]:
query = """
SELECT e.FirstName || ' ' || e.LastName AS employee,
       COUNT(DISTINCT o.OrderID) AS orders,
       ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)), 0) AS revenue,
       ROUND(SUM(od.UnitPrice * od.Quantity * (1 - od.Discount))
             / COUNT(DISTINCT o.OrderID), 0) AS revenue_per_order,
       RANK() OVER (ORDER BY SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) DESC) AS rev_rank
FROM "Order Details" od
JOIN Orders o    ON o.OrderID = od.OrderID
JOIN Employees e ON e.EmployeeID = o.EmployeeID
GROUP BY e.EmployeeID, employee
ORDER BY revenue DESC
"""
pd.read_sql(query, conn)

,employee,orders,revenue,revenue_per_order,rev_rank
0,Margaret Peacock,156,232891.0,1493.0,1
1,Janet Leverling,127,202813.0,1597.0,2
2,Nancy Davolio,123,192108.0,1562.0,3
3,Andrew Fuller,96,166538.0,1735.0,4
4,Laura Callahan,104,126862.0,1220.0,5
5,Robert King,72,124568.0,1730.0,6
6,Anne Dodsworth,43,77308.0,1798.0,7
7,Michael Suyama,67,73913.0,1103.0,8
8,Steven Buchanan,42,68792.0,1638.0,9


## Q6. How many orders shipped late, and does it depend on the shipper?
Late means shipped after the required date. Orders that haven't shipped yet are left out, since you can't call them late.

In [8]:
query = """
SELECT s.CompanyName AS shipper,
       COUNT(*) AS orders,
       SUM(CASE WHEN o.ShippedDate > o.RequiredDate THEN 1 ELSE 0 END) AS late_orders,
       ROUND(100.0 * SUM(CASE WHEN o.ShippedDate > o.RequiredDate THEN 1 ELSE 0 END)
             / COUNT(*), 1) AS late_pct
FROM Orders o
JOIN Shippers s ON s.ShipperID = o.ShipVia
WHERE o.ShippedDate IS NOT NULL
GROUP BY s.CompanyName
ORDER BY late_pct DESC
"""
pd.read_sql(query, conn)

,shipper,orders,late_orders,late_pct
0,United Package,315,16,5.1
1,Speedy Express,245,12,4.9
2,Federal Shipping,249,9,3.6


## Q7. Are discounted orders bigger?
Discounts are stored per line, so a CTE rolls everything up to one row per order first. An order counts as discounted if any line has a discount.

In [9]:
query = """
WITH order_totals AS (
    SELECT od.OrderID,
           MAX(CASE WHEN od.Discount > 0 THEN 1 ELSE 0 END) AS has_discount,
           SUM(od.UnitPrice * od.Quantity * (1 - od.Discount)) AS order_value
    FROM "Order Details" od
    GROUP BY od.OrderID
)
SELECT CASE WHEN has_discount = 1 THEN 'Discounted' ELSE 'No discount' END AS order_type,
       COUNT(*) AS orders,
       ROUND(AVG(order_value), 0) AS avg_order_value
FROM order_totals
GROUP BY has_discount
ORDER BY has_discount DESC
"""
pd.read_sql(query, conn)

,order_type,orders,avg_order_value
0,Discounted,380,1702.0
1,No discount,450,1375.0


## Q8. Which customers have never ordered, or have gone quiet?
`LEFT JOIN` keeps customers with zero orders, and `HAVING` filters after the grouping. The 2 junk 'IT' rows are excluded. 'Quiet' here means no order in the last 4 months of the data.

In [10]:
query = """
SELECT cu.CompanyName,
       cu.Country,
       COUNT(o.OrderID) AS orders,
       MAX(o.OrderDate) AS last_order
FROM Customers cu
LEFT JOIN Orders o ON o.CustomerID = cu.CustomerID
WHERE cu.Country IS NOT NULL            -- excludes the 2 junk 'IT' rows
GROUP BY cu.CustomerID, cu.CompanyName, cu.Country
HAVING COUNT(o.OrderID) = 0
    OR MAX(o.OrderDate) < '2018-01-06'  -- no order in the last 4 months of data
ORDER BY last_order
"""
pd.read_sql(query, conn)

,CompanyName,Country,orders,last_order
0,FISSA Fabrica Inter. Salchichas S.A.,Spain,0,None
1,Paris spécialités,France,0,None
2,Centro comercial Moctezuma,Mexico,1,2016-07-18
3,Lazy K Kountry Store,USA,2,2017-05-22
4,Hungry Coyote Import Store,USA,5,2017-09-08
5,Mère Paillarde,Canada,13,2017-10-30
6,Familia Arquibaldo,Brazil,7,2017-10-31
7,Vins et alcools Chevalier,France,5,2017-11-12
8,GROSELLA-Restaurante,Venezuela,2,2017-12-18
9,Folies gourmandes,France,5,2017-12-22


## What I found
- **A few things carry the business.** Beverages and Dairy are about 40% of revenue, and one product (Côte de Blaye) is about 53% of Beverages. The top 3 customers make up about 25% of revenue and the top 10 about 45%.
- **Country rankings mostly come down to a few big accounts.** The USA and Germany are about 38% of revenue together. Austria is third mostly because of one customer (Ernst Handel, about 82% of its revenue).
- **Growth is strong year on year.** January to April 2018 was about 121% higher than the same months in 2017. The month-to-month numbers are all over the place (anywhere from -37% to +64%), so I'd trust the long-run trend over any single month.
- **Employees win in different ways.** Margaret Peacock has the most revenue (she has the most orders). Anne Dodsworth has the highest revenue per order. Ranking on revenue alone hides that.
- **Late shipping is about the same everywhere.** 4.6% of shipped orders were late overall. The gap between shippers isn't statistically significant (p of about 0.68), so there's no good reason to switch carriers based on this.
- **Discounted orders are about 24% bigger** (roughly 1,702 vs 1,375), but they also have more lines on average, so I can't say discounts cause the bigger orders.
- **Lapsed accounts:** 2 customers never ordered and 9 haven't ordered in four months. The one that stands out is Mère Paillarde, a top-10 customer with 13 orders whose last order was 30 October 2017.

**What I'd do:** look after the biggest accounts and the Beverages line, call lapsed high-value customers like Mère Paillarde first, and test discounts more fairly (like order value per line) before changing any policy.

**Caveats:** Northwind is a made-up company with only 830 orders, so small differences could just be noise. May 2018 is incomplete and was left out of the trend. The 4-month cutoff for 'quiet' customers is crude and would be too strict for anyone who only orders once or twice a year.